# Day 14 Step 0: An explicit tie rule
**The problem (found on Day 13):** 353,571 of 444,431 customers (80%) bought 2+ articles on the same date, and the last-week top 30 has a count tie. Their order came from **row position in the file**, not from a rule. The golden test protects this snapshot, but next week's file has no golden file, so a new row order could silently change who gets which 10 articles.

**The rule, written BEFORE measuring anything:**
- same date -> smaller `article_id` first (repurchase lists)
- same count -> smaller `article_id` first (last-week and age-bucket lists)

It is **arbitrary on purpose**. A "smart" tie-break (e.g. more popular first) would be a modelling change that could move the score, and choosing it after looking would be tuning.

**One copy:** the rule lives only in `src/recsys/components/list_builder.py`; this notebook (and 07, 08) import it.

**Order of work:** 1. measure what changed (no scores) -> 2. re-validate -> 3. test week once, for the record -> 4. re-export the golden files.

The ALS models are **not** retrained: the ALS matrix is a count per (customer, article) with sorted mappings, so it doesn't depend on row order. The saved vectors are reused unchanged.

In [1]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'

import json
import math

import numpy as np
import pandas as pd

from recsys.components.list_builder import (repurchase_lists, last_week_top, age_bucket_top10,
                                            age_to_bucket, fill_up, to_long)

PROJECT = '..'
GOLDEN_FOLDER = PROJECT + '/tests/golden'
FEATURES_FOLDER = PROJECT + '/data/processed/features'
MODELS_FOLDER = PROJECT + '/models'

train = pd.read_parquet(PROJECT + '/data/processed/train.parquet')
val = pd.read_parquet(PROJECT + '/data/processed/val.parquet')
test = pd.read_parquet(PROJECT + '/data/processed/test.parquet')
history_w9 = train
history_w10 = pd.concat([train, val], ignore_index=True)

customers = pd.read_parquet(PROJECT + '/data/RAW/customers.parquet', columns=['customer_id', 'age'])
customers_idx = pd.read_parquet(PROJECT + '/data/processed/customer_join_idx.parquet')
customers = customers.merge(customers_idx, on='customer_id', how='inner', validate='one_to_one')[['customer_join_idx', 'age']]

with open(FEATURES_FOLDER + '/settings.json') as f:
    age_edges = json.load(f)['age_edges']

print('Weeks 0-9 rows: ', len(history_w9), '| expected 2,638,103')
print('Weeks 0-10 rows:', len(history_w10), '| expected 2,869,083')
print('Customers:', len(customers), '| age edges:', age_edges)

Weeks 0-9 rows:  2638103 | expected 2,638,103
Weeks 0-10 rows: 2869083 | expected 2,869,083
Customers: 1371980 | age edges: [16, 20, 25, 30, 40, 50, 60, 100]


## 1. What changed? (lists only, no scores yet)
The OLD lists are the golden files exported on Day 13 (file-order ties). The NEW lists come from the tie-rule code, same history (weeks 0-9, last week = 9).

**Prediction:** repurchase lists mostly **reordered** (same articles, new order); a smaller number get **different articles**, where a same-day tie sat across the 10-item cap. Last-week top 30: only the tied pair can swap.

In [2]:
def long_to_lists(table, key):
    table = table.sort_values([key, 'rank'])
    return table.groupby(key)['article_id'].apply(list).to_dict()

old_repurchase = long_to_lists(pd.read_parquet(GOLDEN_FOLDER + '/golden_repurchase_w9.parquet'), 'customer_join_idx')
new_repurchase = repurchase_lists(history_w9, 10)

changed = [c for c in new_repurchase if new_repurchase[c] != old_repurchase[c]]
reordered = [c for c in changed if set(new_repurchase[c]) == set(old_repurchase[c])]
different_articles = len(changed) - len(reordered)

print('Customers:', len(new_repurchase), '| expected 444,431 (same customers as before)')
print('Repurchase lists changed:', len(changed), '(' + str(round(len(changed) / len(new_repurchase) * 100, 1)) + '%)')
print('  same articles, new order:', len(reordered))
print('  different articles (tie across the 10 cap):', different_articles)

Customers: 444431 | expected 444,431 (same customers as before)
Repurchase lists changed: 0 (0.0%)
  same articles, new order: 0
  different articles (tie across the 10 cap): 0


In [3]:
old_last_week = pd.read_parquet(GOLDEN_FOLDER + '/golden_last_week_w9.parquet').sort_values('rank')['article_id'].to_list()
new_last_week = last_week_top(history_w9, 9, 30)
moved = [rank for rank, (old, new) in enumerate(zip(old_last_week, new_last_week), start=1) if old != new]
print('Last-week top 30: positions changed:', moved, '| same 30 articles:', set(old_last_week) == set(new_last_week))
print('Global top 10 changed:', old_last_week[:10] != new_last_week[:10])

# Age buckets, served exactly like ListBuilder: every bucket filled to 10, bucket 0 = the global list
global_top10 = new_last_week[:10]
bucket_lists = age_bucket_top10(history_w9, customers, 9, age_edges, 10)
new_buckets = {0: global_top10}
for bucket in range(1, len(age_edges)):
    new_buckets[bucket] = fill_up(bucket_lists.get(bucket, []), global_top10)
old_buckets = long_to_lists(pd.read_parquet(GOLDEN_FOLDER + '/golden_age_buckets_w9.parquet'), 'age_bucket')
print('Age-bucket lists changed:', [b for b in new_buckets if new_buckets[b] != old_buckets[b]])

Last-week top 30: positions changed: [] | same 30 articles: True
Global top 10 changed: False


Age-bucket lists changed: []


## 2. Re-validate (weeks 0-9, last week = 9, the Day 7 ALS)
`als_recommend`, the metrics and the final system are copied from notebook 08 (the metrics move to `utils/metrics.py` in Step 2). The only change: the lists come from the imported functions, and cold buckets from `age_to_bucket`.

In [4]:
def recall_at_k(recommended, bought, k):
    hits = 0
    for item in recommended[:k]:
        if item in bought:
            hits = hits + 1
    return hits / len(bought)


def ndcg_at_k(recommended, bought, k):
    dcg = 0
    position = 0
    for item in recommended[:k]:
        position = position + 1
        if item in bought:
            dcg = dcg + 1 / math.log2(position + 1)
    ideal = 0
    for position in range(1, min(len(bought), k) + 1):
        ideal = ideal + 1 / math.log2(position + 1)
    return dcg / ideal


def score(recommendations, bought_lists):
    recall_list = []
    ndcg_list = []
    for recs, bought in zip(recommendations, bought_lists):
        recall_list.append(recall_at_k(list(recs), set(bought), 10))
        ndcg_list.append(ndcg_at_k(list(recs), set(bought), 10))
    return np.array(recall_list), np.array(ndcg_list)


def als_recommend(customer_ids, k, user_factors, item_factors, customer_map, column_to_article):
    rows = np.array([customer_map[c] for c in customer_ids])
    top_columns = []
    for start in range(0, len(rows), 2000):
        chunk_scores = user_factors[rows[start:start + 2000]] @ item_factors.T
        top = np.argpartition(-chunk_scores, k, axis=1)[:, :k]
        order = np.take_along_axis(chunk_scores, top, axis=1).argsort(axis=1)[:, ::-1]
        top_columns.append(np.take_along_axis(top, order, axis=1))
    return column_to_article[np.vstack(top_columns)]


def column_to_article_array(article_map):
    array = np.zeros(len(article_map) + 1, dtype='int64')
    for article, column in article_map.items():
        array[column] = article
    return array


def load_map(path):
    table = pd.read_parquet(path)
    return dict(zip(table['value'], table['index']))


def final_system_recommend(customer_ids, history, week, als_parts):
    # The FROZEN system. Warm: own purchases, then ALS fills (30 candidates). Cold: age-bucket list filled from the global top 10
    own_lists = repurchase_lists(history, 10)
    global_top10 = last_week_top(history, week, 10)
    bucket_lists = age_bucket_top10(history, customers, week, age_edges, 10)

    known = set(history['customer_join_idx'].unique())
    warm_ids = [c for c in customer_ids if c in known]
    cold_ids = [c for c in customer_ids if c not in known]

    recommendations = {}
    for customer, candidates in zip(warm_ids, als_recommend(warm_ids, 30, *als_parts)):
        recommendations[customer] = fill_up(own_lists[customer], candidates)

    cold_ages = customers.set_index('customer_join_idx')['age'].loc[cold_ids]
    cold_buckets = age_to_bucket(cold_ages, age_edges)
    for customer in cold_ids:
        recommendations[customer] = fill_up(bucket_lists.get(cold_buckets[customer], []), global_top10)

    return [recommendations[c] for c in customer_ids]


def report(name, lists, key, old):
    recall, ndcg = score(lists, key['article_id'])
    is_cold = key['is_cold'].to_numpy()
    sizes = [len(set(recs)) for recs in lists]
    print(name, '| lists', len(lists), '| distinct per list', min(sizes), 'to', max(sizes))
    for part, mask in [('all ', np.ones(len(key), bool)), ('warm', ~is_cold), ('cold', is_cold)]:
        print('  ', part, 'recall', round(recall[mask].mean(), 4), ' NDCG', round(ndcg[mask].mean(), 4), '| old', old[part.strip()])
    return recall, ndcg

In [5]:
val_key = val.groupby('customer_join_idx')['article_id'].unique().reset_index()
val_key['is_cold'] = ~val_key['customer_join_idx'].isin(history_w9['customer_join_idx'].unique())
val_ids = list(val_key['customer_join_idx'])

old_als_parts = (np.load(MODELS_FOLDER + '/als/user_factors.npy'),
                 np.load(MODELS_FOLDER + '/als/item_factors.npy'),
                 load_map(FEATURES_FOLDER + '/vocab_customer.parquet'),
                 column_to_article_array(load_map(FEATURES_FOLDER + '/vocab_article.parquet')))

val_lists = final_system_recommend(val_ids, history_w9, 9, old_als_parts)
val_recall, val_ndcg = report('VALIDATION final system', val_lists, val_key,
                              {'all': '0.0495 / 0.0368', 'warm': '0.0631 / 0.0478', 'cold': '0.0216 / 0.0141'})
is_cold_val = val_key['is_cold'].to_numpy()
print('NEW CHAMPION REFERENCE (full precision, for the Step 2 parity check):')
print('  recall all', repr(val_recall.mean()), '| warm', repr(val_recall[~is_cold_val].mean()), '| cold', repr(val_recall[is_cold_val].mean()))
print('  NDCG   all', repr(val_ndcg.mean()), '| warm', repr(val_ndcg[~is_cold_val].mean()), '| cold', repr(val_ndcg[is_cold_val].mean()))

popularity_val = [last_week_top(history_w9, 9, 10)] * len(val_ids)
report('VALIDATION last-week popularity (the gate baseline, Step 3)', popularity_val, val_key,
       {'all': '0.0184 / 0.0118', 'warm': '-', 'cold': '-'});

VALIDATION final system | lists 72019 | distinct per list 10 to 10
   all  recall 0.0495  NDCG 0.0367 | old 0.0495 / 0.0368
   warm recall 0.063  NDCG 0.0477 | old 0.0631 / 0.0478
   cold recall 0.0216  NDCG 0.0141 | old 0.0216 / 0.0141
NEW CHAMPION REFERENCE (full precision, for the Step 2 parity check):
  recall all np.float64(0.04949648088361298) | warm np.float64(0.06302676296272332) | cold np.float64(0.02158250534723137)
  NDCG   all np.float64(0.036730036706961615) | warm np.float64(0.04768484721288037) | cold np.float64(0.014129450061393356)


VALIDATION last-week popularity (the gate baseline, Step 3) | lists 72019 | distinct per list 10 to 10
   all  recall 0.0184  NDCG 0.0118 | old 0.0184 / 0.0118
   warm recall 0.0179  NDCG 0.0116 | old -
   cold recall 0.0194  NDCG 0.0124 | old -


## 3. The test week, once more, for the record
Weeks 0-10, last week = 10, the saved weeks 0-10 ALS (Day 11). The system choice doesn't depend on this number: it was decided on validation.

In [6]:
test_key = test.groupby('customer_join_idx')['article_id'].unique().reset_index()
test_key['is_cold'] = ~test_key['customer_join_idx'].isin(history_w10['customer_join_idx'].unique())
test_ids = list(test_key['customer_join_idx'])

new_als_parts = (np.load(MODELS_FOLDER + '/als_w0_10/user_factors.npy'),
                 np.load(MODELS_FOLDER + '/als_w0_10/item_factors.npy'),
                 load_map(MODELS_FOLDER + '/als_w0_10/customer_map.parquet'),
                 column_to_article_array(load_map(MODELS_FOLDER + '/als_w0_10/article_map.parquet')))

test_lists = final_system_recommend(test_ids, history_w10, 10, new_als_parts)
report('TEST final system', test_lists, test_key,
       {'all': '0.0507 / 0.0370', 'warm': '0.0625 / 0.0467', 'cold': '0.0250 / 0.0157'})
report('TEST last-week popularity', [last_week_top(history_w10, 10, 10)] * len(test_ids), test_key,
       {'all': '0.0222 / 0.0145', 'warm': '0.0221 / 0.0148', 'cold': '0.0224 / 0.0136'});

TEST final system | lists 68984 | distinct per list 10 to 10
   all  recall 0.0508  NDCG 0.0369 | old 0.0507 / 0.0370
   warm recall 0.0625  NDCG 0.0466 | old 0.0625 / 0.0467
   cold recall 0.025  NDCG 0.0157 | old 0.0250 / 0.0157


TEST last-week popularity | lists 68984 | distinct per list 10 to 10
   all  recall 0.0222  NDCG 0.0145 | old 0.0222 / 0.0145
   warm recall 0.0221  NDCG 0.0148 | old 0.0221 / 0.0148
   cold recall 0.0224  NDCG 0.0136 | old 0.0224 / 0.0136


## 4. Re-export the golden files from the tie-rule code
Same long format as Day 13. After this, `python main.py` + `pytest` must pass again: the pipeline and this notebook now run **the same imported functions**, and the golden files pin their output so any future drift fails a test.

In [7]:
golden_repurchase = to_long(new_repurchase, 'customer_join_idx')
golden_last_week = pd.DataFrame({'rank': range(1, len(new_last_week) + 1), 'article_id': new_last_week})
golden_age_buckets = to_long(new_buckets, 'age_bucket')

for table in (golden_repurchase, golden_last_week, golden_age_buckets):
    for column in ('customer_join_idx', 'age_bucket', 'rank', 'article_id'):
        if column in table:
            table[column] = table[column].astype('int32')

golden_repurchase.to_parquet(GOLDEN_FOLDER + '/golden_repurchase_w9.parquet', index=False)
golden_last_week.to_parquet(GOLDEN_FOLDER + '/golden_last_week_w9.parquet', index=False)
golden_age_buckets.to_parquet(GOLDEN_FOLDER + '/golden_age_buckets_w9.parquet', index=False)

print('Repurchase:', golden_repurchase['customer_join_idx'].nunique(), 'customers,', len(golden_repurchase), 'rows | expected 444,431 / 2,091,980')
print('Last week: ', len(golden_last_week), 'rows | expected 30')
print('Age buckets:', len(golden_age_buckets), 'rows | expected 80')

Repurchase: 444431 customers, 2091980 rows | expected 444,431 / 2,091,980
Last week:  30 rows | expected 30
Age buckets: 80 rows | expected 80
